### Pydantic Basics : Creating and Using Models
Pydantic models are the foundation of data validation in Python. They use Python type annotations to define the structure and validate data at runtime.

In [15]:
from pydantic import BaseModel

In [16]:
from dataclasses import dataclass

@dataclass
class Person():
    name:str
    age:int
    city:str

person1=Person(name="Mathura", age=50, city="Paris")
person2=Person(name="Mathura", age=50, city=12) # no pydantic thus no data validation thus no error

print(person1)
print(person2)

Person(name='Mathura', age=50, city='Paris')
Person(name='Mathura', age=50, city=12)


In [17]:
# simple model that inheritates BaseModel to see how does the data validation happen
class Person1(BaseModel):
    name:str
    age:int
    city:str

person=Person1(name="Mathura", age=50, city="Paris")
print(person)

name='Mathura' age=50 city='Paris'


In [18]:
type(person)

__main__.Person1

In [21]:
# the result is the same as using dataclass but if we create another object with an int instead of a str :
person3=Person1(name="Mathura", age=50, city=12)
print(person3)
# because of pydantic, it says "ValidationError: 1 validation error for Person" which I don't get with dataclass (no data validation)

ValidationError: 1 validation error for Person1
city
  Input should be a valid string [type=string_type, input_value=12, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type

### 2. Model with Optional Fields
Add optionnal fields using Python's Optional type :

In [22]:
from typing import Optional
class Employee(BaseModel):
    id: int
    name: str
    department: str
    salary: Optional[float] = None # Optional with default value
    is_active: Optional[bool] = True # Optional with default True


In [26]:
# Examples with and without optional fields

employee1 = Employee(id=1, name="John", department="IT")
print(employee1) # id=1, name="John", department="IT", salary=None is_active=True

id=1 name='John' department='IT' salary=None is_active=True


In [28]:
employee2 = Employee(id=2, name="Jane", department="HR", salary=60000, is_active=False)
print(employee2)
# automatic casting of int to float (salary)

id=2 name='Jane' department='HR' salary=60000.0 is_active=False


Definition :
- Optional[type]: Indicates the field can be None

- Default value (= None or = True): Makes the field optional

- Required fields must still be provided

- Pydantic validates types even for optional fields when values are provided

In [30]:
# Model with list values / list fields
from pydantic import BaseModel
from typing import List

class Classroom(BaseModel):
    room_num: str
    students: List[str] # list of strings values
    capacity: int

In [31]:
# create a classroom
classroom = Classroom(
    room_num="A101",
    students=["Alice", "Bob", "Charlie"],
    capacity=30
)
print(classroom)

# if i give () instead of [] => casting
# if i give a dictionnary => data validation error

room_num='A101' students=['Alice', 'Bob', 'Charlie'] capacity=30


In [32]:
# list of str + int instead of list of str
try:
    invalid_val=Classroom(room_num="A1", students=["Mathura", 123], capacity=30)
except ValueError as e:
    print(e)

1 validation error for Classroom
students.1
  Input should be a valid string [type=string_type, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type


#### 3. Model with Nested Models
Create complex structures with nested models:

In [36]:
from pydantic import BaseModel

class Address(BaseModel):
    street: str
    city: str
    zip_code: int

class Customer(BaseModel):
    customer_id: int
    name: str
    address: Address # nested model

# Create a customer with nested model
customer = Customer(
    customer_id= 1,
    name= "Emma",
    address={"street": "123 rue du Moulin", "city": "Paris", "zip_code": "75000"}
)
print(customer)

# type casting : str into int for zip_code

customer_id=1 name='Emma' address=Address(street='123 rue du Moulin', city='Paris', zip_code=75000)


#### Pydantic Fields : Customization and Constraints

The Field function in Pydantic enhances model fields beyond basic type hints by allowing to specify **validation rules**, default values, aliases, and more.

In [39]:
from pydantic import BaseModel, Field
class Item(BaseModel):
    name:str= Field(min_length=2, max_length=50)
    price:float= Field(gt=0,le=1000) # should be greater than 0 and less or equal to 1000
    quantity:int=Field(ge=0)

# Valid instance
item = Item(name="Book", price=29.00, quantity=10)

print(item)

# N.B. : data validation is really fast !

name='Book' price=29.0 quantity=10


In [40]:
from pydantic import BaseModel, Field

class User (BaseModel):
    username: str = Field(..., description="Unique username for the user")
    age: int = Field(default=18, description="User age, defaults to 18")
    email: str = Field(default_factory=lambda: "user@example.com", description="Default email address")

# Examples
user1 = User(username="alice")
print(user1) # username='alice' age=18 email='user@example.com'

user2 = User (username="bob", age=25, email="bob@domain.com")
print (user2) #username= bob age=25 email= bob@domain.com

username='alice' age=18 email='user@example.com'
username='bob' age=25 email='bob@domain.com'


In [42]:
print(User.model_json_schema())

{'properties': {'username': {'description': 'Unique username for the user', 'title': 'Username', 'type': 'string'}, 'age': {'default': 18, 'description': 'User age, defaults to 18', 'title': 'Age', 'type': 'integer'}, 'email': {'description': 'Default email address', 'title': 'Email', 'type': 'string'}}, 'required': ['username'], 'title': 'User', 'type': 'object'}
